# Programming Assignment 1: Text Preprocessing & EDA
**Twitter Entity Sentiment – preprocessing, POS analysis and visualization**

## 📋 Assignment Information

**Course:** Unstructured Data Analysis (2026-2)
**Assignment:** Programming Assignment 1 – Text Preprocessing
**Released:** September 21, 2026 (Week 3 – Programming)
**Deadline:** September 28, 2026 (Mon), 23:59
**Total:** 100 points

## 🎯 Objectives

In this assignment you will apply the preprocessing pipeline from the Week 3 practice to a real Twitter dataset:

- Clean raw tweets (lowercasing, URL / symbol removal) and tokenize them with a regular expression
- Remove stop-words (NLTK list + custom list)
- POS-tag the tokens and perform **POS-aware lemmatization**
- Compare the **POS distribution** and the **most frequent words** of Positive / Neutral / Negative tweets
- Visualize the results with bar charts and word clouds

## 📝 Instructions

1. `File ▸ Save a copy in Drive`, then rename the notebook to **`Assignment_1_YourName_StudentID.ipynb`** (e.g. `Assignment_1_HojinSon_2026123456.ipynb`).
2. Fill in every `________` blank and every `# TODO` line. Do **not** change the code outside the TODO blocks, and do **not** change variable names – the grader runs your notebook.
3. Every code cell must run without errors from top to bottom (`Runtime ▸ Restart and run all`). Keep the outputs in the notebook when you submit.
4. Write your answers to the discussion questions in the markdown cells at the end.
5. Submit the `.ipynb` file to **"Programming Assignment 1 – Text Preprocessing"** under *Assignments*.

**Grading** – points are given per TODO item (see the table). Partial credit is given when the logic is correct but the output differs slightly.

| Part | Content | Points |
|---|---|---|
| 1 | Data loading & cleaning (TODO 1–3) | 15 |
| 2 | Tokenization & stop-words (TODO 4–5) | 20 |
| 3 | POS tagging & lemmatization (TODO 6–8) | 25 |
| 4 | Word-frequency analysis (TODO 9–10) | 20 |
| 5 | Word clouds (TODO 11–12) | 10 |
| 6 | Discussion questions (Q1–Q2) | 10 |
| | **Total** | **100** |

## 📂 Dataset – Twitter Entity Sentiment Analysis

Source: https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis

| column | description |
|---|---|
| `idx` | tweet id (the same tweet can appear several times with small variations → we keep only the first) |
| `Entity` | the game / company the tweet is about (Borderlands, Microsoft, …) |
| `Sentiment` | `Positive`, `Negative`, `Neutral`, `Irrelevant` |
| `Review` | the tweet text |

In this assignment we **drop `Irrelevant`** tweets and analyse the three remaining classes.

## &nbsp;0. Setup
*This section is complete – no TODO items here.*

In [ ]:
import nltk
for r in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
          'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(r, quiet=True)
!pip install -q wordcloud

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import FreqDist, bigrams
from wordcloud import WordCloud

import warnings
warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

SENTIMENTS = ['Positive', 'Neutral', 'Negative']
COLORS = {'Positive': '#2a9d8f', 'Neutral': '#8d99ae', 'Negative': '#e76f51'}

## &nbsp;1. Data Loading & Cleaning (15 pts)
The dataset is loaded directly from the course GitHub repository, so you do **not** need Google Drive for this assignment.
(The same file is in the repository under `week03-text-preprocessing/data/twcs.csv` if you prefer to download it.)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week03-text-preprocessing/data/twcs.csv"
data = pd.read_csv(DATA_URL)

# If the download fails, download twcs.csv from the repository, drag it into the
# Files panel on the left, and use:  data = pd.read_csv('twcs.csv')

print(data.shape)
data.head()

### TODO 1: Remove duplicated tweets (5 points)

The same `idx` appears several times. Keep only the **first** row of every `idx`.

**Expected output:** `(12447, 4)`

In [ ]:
# TODO 1: drop duplicated idx, keep the first occurrence
data_unique = data.drop_duplicates(subset=________, keep=________)

print(data_unique.shape)

### TODO 2: Drop `Irrelevant` tweets (5 points)

Keep only the rows whose `Sentiment` is **not** `Irrelevant`, then reset the index (`drop=True`).

**Expected output:** shape `(10282, 4)` and the class counts `Negative 3757 / Positive 3472 / Neutral 3053`.

In [ ]:
# TODO 2: remove Irrelevant tweets and reset the index
df = data_unique[data_unique['Sentiment'] ________ 'Irrelevant'].reset_index(drop=True)

print(df.shape)
print(df['Sentiment'].value_counts())

### TODO 3: Clean the tweet text (5 points)

Complete `clean_text()` so that it
1. converts the text to a **lowercase string**,
2. removes **URLs** – anything starting with `http` or `www`, and Twitter picture links that contain `.com` (e.g. `pic.twitter.com/abc`) – up to the next white space,
3. replaces every character that is **not** a lowercase letter, an apostrophe or white space with a space.

**Expected output** for the test tweet: `"i loooove borderlands   check this out      it's awesome"` (the number of spaces may differ).

In [ ]:
def clean_text(d):
    d = str(d).________()                              # 1) lowercase string
    d = re.sub(r"________", " ", d)                    # 2) remove URLs  (hint: http\S+ | www\S+ | \S+\.com\S*)
    d = re.sub(r"________", " ", d)                    # 3) keep only a-z, apostrophe and white space
    return d

test = "I loooove @Borderlands!!! Check this out: https://t.co/xyz123 pic.twitter.com/abc <3 it's AWESOME 10/10"
print(repr(clean_text(test)))

df['clean'] = df['Review'].apply(clean_text)
df[['Sentiment', 'Review', 'clean']].head()

## &nbsp;2. Tokenization & Stop-words (20 pts)

### TODO 4: Regular-expression tokenization (8 points)

Create a `RegexpTokenizer` that returns tokens made of **lowercase letters or apostrophes with at least 2 characters**
(so single letters such as `i` / `u` are dropped) and apply it to the `clean` column.

**Expected output:** the first tweet becomes `['im', 'getting', 'on', 'borderlands', 'and', 'will', 'murder', 'you', 'all']`

In [ ]:
# TODO 4: tokenizer pattern – letters / apostrophes, 2 or more characters
tokenizer = RegexpTokenizer(r"________")

df['tokens'] = df['clean'].apply(________)

print(df['tokens'][0])
print('Total tokens:', df['tokens'].apply(len).sum())

### TODO 5: Stop-word removal (12 points)

1. Build `stop_words` = NLTK English stop-words **plus** the custom list `CUSTOM_STOPS` below (tweet slang and words that appear in every class).
2. Complete `remove_stopwords()` and create the column `tokens_clean`.
3. Print the total number of tokens **per sentiment** after stop-word removal.

**Expected output:** `Positive 30,743 / Neutral 32,667 / Negative 37,650` tokens.

In [ ]:
CUSTOM_STOPS = ['im', 'ive', 'dont', 'cant', 'thats', 'like', 'get', 'got', 'one', 'game', 'games', 'amp']

# TODO 5-1: NLTK English stop-words + custom stop-words (use a set)
stop_words = set(stopwords.words(________)) | set(________)

# TODO 5-2: keep only the tokens that are NOT stop-words
def remove_stopwords(tokens):
    return [t for t in tokens if ________]

df['tokens_clean'] = df['tokens'].apply(remove_stopwords)

# TODO 5-3: total number of tokens per sentiment
token_counts = df.groupby('Sentiment')['tokens_clean'].apply(lambda s: ________)
print(token_counts)

## &nbsp;3. POS Tagging & Lemmatization (25 pts)

### TODO 6: Penn Treebank → WordNet POS mapping (10 points)

`nltk.pos_tag` returns Penn Treebank tags (`NN`, `VBD`, `JJ`, `RB`, …) but `WordNetLemmatizer` needs `wordnet.NOUN / VERB / ADJ / ADV`.
Complete the mapping using the **first letter** of the tag. Return `None` for any other tag.

In [ ]:
def penn_to_wordnet(tag):
    if tag.startswith('J'):
        return ________
    elif tag.startswith('V'):
        return ________
    elif tag.startswith('N'):
        return ________
    elif tag.startswith('R'):
        return ________
    else:
        return None

# quick test – expected: n v a r None
print(penn_to_wordnet('NNS'), penn_to_wordnet('VBD'), penn_to_wordnet('JJR'), penn_to_wordnet('RB'), penn_to_wordnet('DT'))

### TODO 7: POS-aware lemmatization (10 points)

Complete `lemmatize_tokens()`:
1. POS-tag the token list with `nltk.pos_tag`,
2. convert each tag with `penn_to_wordnet`,
3. lemmatize with the WordNet POS when available, otherwise keep the token as it is.
The function returns **two** lists: the lemmas and the Penn tags (we need the tags for TODO 8).

**Expected output** for the test tokens: `['play', 'best', 'game', 'yesterday', 'friend']`

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    tagged = nltk.________(tokens)                       # 1) POS tagging
    lemmas, tags = [], []
    for word, tag in tagged:
        wn_tag = ________(tag)                           # 2) Penn → WordNet
        if wn_tag is None:
            lemmas.append(________)                      # 3a) no WordNet POS → keep the token
        else:
            lemmas.append(lemmatizer.lemmatize(________, ________))   # 3b) lemmatize with POS
        tags.append(tag)
    return lemmas, tags

print(lemmatize_tokens(['played', 'best', 'games', 'yesterday', 'friends'])[0])

results = df['tokens_clean'].apply(lemmatize_tokens)     # this takes ~1 minute
df['lemmas'] = results.apply(lambda x: x[0])
df['tags'] = results.apply(lambda x: x[1])
df[['Sentiment', 'tokens_clean', 'lemmas']].head()

### TODO 8: POS distribution per sentiment (5 points)

Group the Penn tags into 5 categories (`noun`, `verb`, `adjective`, `adverb`, `other`) and compute the **percentage** of each category for every sentiment.
Complete the two blanks, then run the plotting cell.

In [ ]:
def pos_category(tag):
    if tag.startswith('N'):   return 'noun'
    elif tag.startswith('V'): return 'verb'
    elif tag.startswith('J'): return 'adjective'
    elif tag.startswith('R'): return 'adverb'
    else:                     return 'other'

pos_table = {}
for s in SENTIMENTS:
    all_tags = [tag for tags in df.loc[df['Sentiment'] == s, 'tags'] for tag in tags]   # flatten
    cats = Counter(________(t) for t in all_tags)                                     # TODO: map every tag to its category
    total = sum(cats.values())
    pos_table[s] = {c: round(cats[c] / ________ * 100, 2) for c in ['noun', 'verb', 'adjective', 'adverb', 'other']}  # TODO: percentage

pos_df = pd.DataFrame(pos_table)
pos_df

In [ ]:
# Grouped bar chart of the POS distribution (complete – no TODO)
ax = pos_df.plot(kind='bar', figsize=(9, 4), color=[COLORS[s] for s in SENTIMENTS], rot=0)
ax.set_ylabel('% of tokens'); ax.set_title('POS distribution by sentiment')
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f', fontsize=8)
plt.show()

## &nbsp;4. Word-Frequency Analysis (20 pts)

### TODO 9: Top-15 words per sentiment (10 points)

For every sentiment, flatten the `lemmas` column into one list, build a `FreqDist`, and store it in `freq_dict`.
Print the 15 most common words of each class.

In [ ]:
freq_dict = {}
for s in SENTIMENTS:
    words = [w for lemmas in df.loc[df['Sentiment'] == s, ________] for w in lemmas]   # TODO: flatten the lemma lists
    freq_dict[s] = ________(words)                                                    # TODO: frequency distribution
    print(f"[{s}] top 15:", freq_dict[s].________(15))                                # TODO: 15 most common

### TODO 10: Horizontal bar charts (10 points)

Draw three horizontal bar charts side by side (`1 × 3` subplots) showing the top-15 words of each sentiment.
The most frequent word must be at the **top** of each chart.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, s in zip(axes, SENTIMENTS):
    top = freq_dict[s].most_common(15)
    words = [w for w, c in top][________]        # TODO: reverse the order so the most frequent word is on top
    counts = [c for w, c in top][________]
    ax.________(words, counts, color=COLORS[s])  # TODO: horizontal bar chart
    ax.set_title(f'{s} – top 15 words')
plt.tight_layout()
plt.show()

## &nbsp;5. Word Clouds (10 pts)

### TODO 11: One word cloud per sentiment (6 points)

Use `generate_from_frequencies` with the `FreqDist` objects from TODO 9.
Settings: `width=600, height=400, background_color='white', max_words=80, collocations=False`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, s in zip(axes, SENTIMENTS):
    wc = WordCloud(width=600, height=400, background_color=________, max_words=________,
                   collocations=________).________(freq_dict[s])     # TODO
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(s, fontsize=14)
    ax.axis('off')
plt.show()

### TODO 12: Bigram word cloud for Negative tweets (4 points)

Build bigrams **within each tweet** (so that the last word of one tweet is not paired with the first word of the next),
join the two words with `_`, count them, and draw a word cloud from the counts. Print the 10 most common bigrams.

In [ ]:
neg_bigrams = []
for lemmas in df.loc[df['Sentiment'] == 'Negative', 'lemmas']:
    neg_bigrams += ["_".join(bg) for bg in ________(lemmas)]      # TODO: bigrams of one tweet

bigram_freq = Counter(neg_bigrams)
print(bigram_freq.most_common(10))

wc = WordCloud(width=800, height=400, background_color='white', max_words=60).________(bigram_freq)   # TODO
plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear'); plt.axis('off'); plt.title('Negative – bigrams')
plt.show()

## &nbsp;6. Discussion Questions (10 pts)
Write your answers in the markdown cells below (3–5 sentences each). Base your answers on **your own outputs** above.

### Question 1 (5 points) – POS distribution
Compare the POS distribution of the three sentiment classes (TODO 8). Which part of speech differs most between Positive/Negative and Neutral tweets, and how would you explain this from the way people express opinions?

*[Write your answer here]*

### Question 2 (5 points) – Effect of preprocessing
Look at the top-15 words and word clouds. (a) Give two concrete examples of how **stop-word removal** or **lemmatization** changed the result (e.g. a word that would otherwise dominate all three classes, or inflected forms that were merged). (b) Suggest one additional preprocessing step that would make the sentiment-specific words stand out more clearly, and explain why.

*[Write your answer here]*